In [ ]:

# 1. Configuração e Dependências
import os
import sys

try:
    from google.colab import drive
    IN_COLAB = True
    print("Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally")

if IN_COLAB:
    if not os.path.exists('/content/ufc-easytpp'):
        !git clone https://github.com/hugoramos/ufc-easytpp.git /content/ufc-easytpp
    
    !pip install -q torch numpy matplotlib pandas datasets pyyaml transformers gdown

    %cd /content/ufc-easytpp
    if '/content/ufc-easytpp' not in sys.path:
        sys.path.append('/content/ufc-easytpp')

else:
    local_path = '/Users/hugoramossoares/Sites/EasyTemporalPointProcess'
    if os.path.exists(local_path):
        os.chdir(local_path)
        if local_path not in sys.path:
            sys.path.append(local_path)

import torch
import numpy as np
import matplotlib.pyplot as plt
import pickle
from easy_tpp.model.torch_model.torch_nhp import NHP
from easy_tpp.model.torch_model.torch_thp import THP
print("Bibliotecas carregadas.")


In [ ]:

# 2. Baixar MIMIC-II do Google Drive (Fold 1)
# ID da pasta: 1DJZjYv1eWcmK55xmRk4jVRSB1Alx3vY4

# Usaremos gdown para baixar a pasta inteira
# A pasta contém subpastas 'fold1', 'fold2', etc.
# Vamos focar no 'fold1' para este experimento.

import gdown
import shutil

DATA_DIR = 'data/mimic_folds'
os.makedirs(DATA_DIR, exist_ok=True)

# URL da pasta principal do Drive
folder_url = 'https://drive.google.com/drive/folders/1DJZjYv1eWcmK55xmRk4jVRSB1Alx3vY4'

print("Baixando pasta do Google Drive (pode levar alguns minutos)...")
# gdown --folder baixa a estrutura recursivamente
# Se falhar no Colab por limites de cota, tente baixar manualmente e colocar no Drive pessoal

try:
    # Tenta baixar a pasta 'fold1' especificamente se possível, ou a raiz
    # Como o ID é da raiz com vários folds, vamos tentar baixar tudo
    if not os.path.exists(os.path.join(DATA_DIR, 'fold1')):
        !gdown --folder {folder_url} -O {DATA_DIR}
    else:
        print("Pasta já existe.")
except Exception as e:
    print(f"Erro no download automático: {e}")
    print("Se o gdown falhar, por favor baixe manualmente o fold1 e faça upload para o Colab.")

print("\nConteúdo baixado:")
!ls -R {DATA_DIR}


In [ ]:

# 3. Carregar Dados (Fold 1)
# O formato exato dos pkls neste drive precisa ser verificado.
# Geralmente é uma lista de dicionários ou lista de listas.

def load_pickle(path):
    with open(path, 'rb') as f:
        # Tenta carregar com encoding latin1 (comum em python2->3)
        try:
            return pickle.load(f, encoding='latin1')
        except:
            return pickle.load(f)

def inspect_data(data, name="Data"):
    print(f"\n--- Inspecionando {name} ---")
    print(f"Tipo: {type(data)}")
    if isinstance(data, list):
        print(f"Tamanho da lista: {len(data)}")
        if len(data) > 0:
            sample = data[0]
            print(f"Amostra[0] tipo: {type(sample)}")
            if isinstance(sample, dict):
                print(f"Chaves: {sample.keys()}")
            elif isinstance(sample, list):
                print(f"Exemplo (lista): {sample[:5]}...")
            elif hasattr(sample, '__dict__'):
                print(f"Atributos: {sample.__dict__.keys()}")
    elif isinstance(data, dict):
        print(f"Chaves: {data.keys()}")

# Caminho para fold1
base_path = os.path.join(DATA_DIR, 'data_mimic', 'fold1') 
# Ajuste conforme o nome da pasta baixada pelo gdown (pode ser 'data_mimic' ou o nome do ID)
if not os.path.exists(base_path):
    # Tentar encontrar onde baixou
    found = False
    for root, dirs, files in os.walk(DATA_DIR):
        if 'fold1' in dirs:
            base_path = os.path.join(root, 'fold1')
            found = True
            break
    if not found:
        print("AVISO: Pasta fold1 não encontrada automaticamente. Verifique o output do ls.")

train_path = os.path.join(base_path, 'train.pkl')
dev_path = os.path.join(base_path, 'dev.pkl')
test_path = os.path.join(base_path, 'test.pkl')

print(f"Carregando de: {base_path}")

try:
    train_data_raw = load_pickle(train_path)
    dev_data_raw = load_pickle(dev_path)
    test_data_raw = load_pickle(test_path)
    
    inspect_data(train_data_raw, "Treino Raw")
except Exception as e:
    print(f"Erro ao carregar arquivos: {e}")
    # Criar dummy data se falhar para o notebook não quebrar na demo
    train_data_raw = []

# --- Adaptador de Formato ---
# Se o formato for diferente do esperado pelo EasyTPP ({'time_since_start':..., 'type_event':...}),
# precisamos converter.
# O formato comum em Mei/Eisner (NHP) é: lista de dicts com 'time_since_start', 'time_since_last_event', 'type_event'
# Se for isso, ótimo. Se for lista de listas [(t, k), ...], converteremos.

def standardize_data(raw_data):
    processed = []
    if not raw_data: return []
    
    sample = raw_data[0]
    
    # Caso 1: Já é o formato EasyTPP
    if isinstance(sample, dict) and 'time_since_start' in sample:
        return raw_data
        
    # Caso 2: Lista de dicts com chaves diferentes (ex: 'time', 'event')
    if isinstance(sample, dict):
        # Tentar mapear
        print("Detectado dict com chaves customizadas. Tentando adaptar...")
        # Adicione lógica se necessário
        return raw_data
        
    # Caso 3: Lista de listas de dicts (ex: [{'time_since_start':...}])
    # (Às vezes o pickle carrega aninhado)
    
    return raw_data

train_data = standardize_data(train_data_raw)
dev_data = standardize_data(dev_data_raw)
test_data = standardize_data(test_data_raw)

# Estatísticas
def get_stats(data):
    if not data: return 0, 1.0
    
    lens = [len(x['time_since_start']) for x in data]
    all_types = [t for x in data for t in x['type_event']]
    all_deltas = [d for x in data for d in x['time_since_last_event'] if d > 0]
    
    num_types = max(all_types) + 1 if all_types else 0
    time_scale = np.mean(all_deltas) if all_deltas else 1.0
    
    print(f"Num Seqs: {len(data)}")
    print(f"Avg Len: {np.mean(lens):.1f}")
    print(f"Num Types: {num_types}")
    print(f"Time Scale: {time_scale:.4f}")
    return num_types, time_scale

print("\nEstatísticas Treino:")
NUM_TYPES, TIME_SCALE = get_stats(train_data)


In [ ]:

# 4. Configuração do Modelo e Collate
# (Mantendo a normalização temporal que é crucial)

def collate_fn_factory(time_scale):
    def collate_fn(batch_list):
        time_seqs = []
        time_delta_seqs = []
        type_seqs = []
        max_len = 0
        
        for item in batch_list:
            ts = item['time_since_start']
            td = item['time_since_last_event']
            ev = item['type_event']
            
            if len(ts) > max_len: max_len = len(ts)
            
            # Normalização Temporal
            ts_norm = (torch.tensor(ts, dtype=torch.float64) - ts[0]) / time_scale
            td_norm = torch.tensor(td, dtype=torch.float64) / time_scale
            
            time_seqs.append(ts_norm.float())
            time_delta_seqs.append(td_norm.float())
            type_seqs.append(torch.tensor(ev, dtype=torch.long))
            
        batch_size = len(batch_list)
        pad_time = torch.zeros(batch_size, max_len)
        pad_delta = torch.zeros(batch_size, max_len)
        pad_type = torch.zeros(batch_size, max_len, dtype=torch.long)
        mask = torch.zeros(batch_size, max_len)
        attn_mask = torch.zeros(batch_size, max_len, max_len)
        
        for i in range(batch_size):
            l = len(time_seqs[i])
            pad_time[i, :l] = time_seqs[i]
            pad_delta[i, :l] = time_delta_seqs[i]
            pad_type[i, :l] = type_seqs[i]
            mask[i, :l] = 1
            
            # Causal + Pad Mask
            causal = torch.triu(torch.ones(max_len, max_len), diagonal=1)
            causal[:, l:] = 1
            causal[l:, :] = 1
            attn_mask[i] = causal
            
        return (pad_time, pad_delta, pad_type, mask, attn_mask)
    return collate_fn

class ModelConfig:
    def __init__(self, num_types, hidden_size=64):
        self.num_event_types = num_types
        self.num_event_types_pad = num_types + 1
        self.pad_token_id = num_types
        self.hidden_size = hidden_size
        self.time_emb_size = hidden_size
        self.num_layers = 2
        self.num_heads = 4
        self.dropout_rate = 0.1
        self.use_ln = True
        self.gpu = 0 if torch.cuda.is_available() else -1
        self.thinning = type('Thinning',(),{'num_sample':100,'num_exp':500,'over_sample_rate':10.0,'patience_counter':5,'num_samples_boundary':20,'dtime_max':5.0})()
        self.model_specs = {'beta': 1.0, 'bias': True}


In [ ]:

# 5. Loop de Treinamento e Avaliação

def compute_metrics(model, test_ds, collate_fn):
    model.eval()
    total_acc = 0
    total_rmse = 0
    total_ev = 0
    subset = test_ds[:200]
    
    with torch.no_grad():
        for i in range(0, len(subset), 32):
            batch = collate_fn(subset[i:i+32])
            _, target_delta, target_type, target_mask, _ = batch
            
            dtimes_pred, types_pred = model.predict_one_step_at_every_event(batch)
            
            # Align (predict t+1 from t)
            # Targets start from index 1
            tgt_type = target_type[:, 1:]
            tgt_delta = target_delta[:, 1:]
            mask = target_mask[:, 1:]
            
            acc = (types_pred == tgt_type) * mask
            total_acc += acc.sum().item()
            
            se = ((dtimes_pred - tgt_delta) ** 2) * mask
            total_rmse += se.sum().item()
            
            total_ev += mask.sum().item()
            
    return total_acc / (total_ev+1e-9), np.sqrt(total_rmse / (total_ev+1e-9))

def train(model_class, name, config, train_ds, test_ds, time_scale, epochs=10):
    print(f"\n>>> Treinando {name}...")
    model = model_class(config)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    collate = collate_fn_factory(time_scale)
    
    hist = []
    
    for ep in range(epochs):
        model.train()
        total_loss = 0
        total_n = 0
        
        # Use simple iteration for demo
        indices = np.random.permutation(len(train_ds))
        # Limit if too large
        if len(indices) > 2000: indices = indices[:2000]
        
        for i in range(0, len(indices), 64):
            batch_list = [train_ds[k] for k in indices[i:i+64]]
            batch = collate(batch_list)
            
            opt.zero_grad()
            loss, n = model.loglike_loss(batch)
            
            if torch.isnan(loss): 
                # print("NaN")
                continue
                
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            
            total_loss += loss.item()
            total_n += n
            
        train_nll = total_loss / (total_n + 1e-9)
        
        # Validation
        model.eval()
        val_loss = 0
        val_n = 0
        with torch.no_grad():
            subset_val = test_ds[:200]
            batch = collate(subset_val)
            l, n = model.loglike_loss(batch)
            val_loss += l.item()
            val_n += n
            
        val_nll = val_loss / (val_n + 1e-9)
        
        hist.append(val_nll)
        print(f"  Ep {ep+1} | Train NLL: {train_nll:.4f} | Val NLL: {val_nll:.4f}")
        
    acc, rmse = compute_metrics(model, test_ds, collate)
    print(f"  >>> Final {name}: Acc={acc:.4f}, RMSE={rmse:.4f}")
    return hist, acc, rmse, model

# Executar Comparação
if 'NUM_TYPES' in locals() and NUM_TYPES > 0:
    config = ModelConfig(NUM_TYPES, hidden_size=64)
    
    hist_nhp, acc_nhp, rmse_nhp, m_nhp = train(NHP, "NHP (RNN)", config, train_data, test_data, TIME_SCALE, epochs=15)
    hist_thp, acc_thp, rmse_thp, m_thp = train(THP, "THP (Transformer)", config, train_data, test_data, TIME_SCALE, epochs=15)
    
    # Plot NLL
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(hist_nhp, label=f'NHP (Final NLL={hist_nhp[-1]:.3f})')
    plt.plot(hist_thp, label=f'THP (Final NLL={hist_thp[-1]:.3f})')
    plt.title('NLL Convergence')
    plt.xlabel('Epoch')
    plt.ylabel('Negative Log Likelihood')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Plot Metrics
    plt.subplot(1, 2, 2)
    metrics = ['Accuracy', 'RMSE']
    x = np.arange(len(metrics))
    width = 0.35
    
    nhp_vals = [acc_nhp, rmse_nhp]
    thp_vals = [acc_thp, rmse_thp]
    
    plt.bar(x - width/2, nhp_vals, width, label='NHP')
    plt.bar(x + width/2, thp_vals, width, label='THP')
    plt.xticks(x, metrics)
    plt.title('Final Metrics')
    plt.legend()
    plt.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
else:
    print("Não foi possível carregar os dados para treinar. Verifique o download do Drive.")
